In [1]:
import pandas, random, time, os, sys, math
import numpy as np
from sklearn import preprocessing
from rdkit import Chem
from mordred import Calculator, descriptors
from warnings import simplefilter
from sklearn.exceptions import ConvergenceWarning
simplefilter("ignore", category=ConvergenceWarning)

# Modules created as a part of this publication
#import hpctools
import peptideutils as pu
import prescreen
import active_learning as al
    
#hpc = hpctools.hpctools()

def normalize_pandas_data(data):
    min_max_scaler = preprocessing.MinMaxScaler(feature_range=(-1, 1))
    new_data = min_max_scaler.fit_transform(data)
    return pandas.DataFrame(new_data, index=data.index, columns=data.columns)

def PredictHighestUnknown(model, X_test, peptides, cutoff=-1):
    p = model.predict(X_test)
    if cutoff != -1:
        all_ps = list(p)
        p = p[p < cutoff]
        if len(p) == 0:
            return False, False, False
        else:
            p = max(p)
            index = all_ps.index(p)
            return peptides[index], index, p
    else:
        all_ps = list(p)
        p = max(p)
        index = all_ps.index(p)
        return peptides[index], index, p

def readin(fpath):
    f = open(fpath)
    content = f.read()
    f.close()
    return content

def juparse(fpath):
    content = readin(fpath)
    peptides = []
    for line in content.split("\n"):
        line = line.split()
        if len(line) >= 4:
            pep = "".join(line[2:])
            pep = pu.translate1to3(pep)
            peptides.append(pep)
    return peptides

def makepeptide(peptide):
    """
    
    INFO       Chain termini will be charged
    
    """
    mol = Chem.MolFromSequence(peptide)
    mol.GetAtomWithIdx(0).SetFormalCharge(1)
    mol.GetAtomWithIdx(len(list(mol.GetAtoms()))-1).SetFormalCharge(-1)
    return mol

def gen(peptides, keys, CPUS):
    """
    Parameters
    ----------
    peptide : list
        list of strings of single capital letter amino acid code
    keys : list
        mordred parameters to return

    Returns
    -------
    Mordred parameters

    """
    if os.path.exists("Tripeptide_Mordred.csv") and len(peptides[0]) == 3:
        peptides = [pu.translate1to3(x) for x in peptides]
        df = pandas.read_csv("Tripeptide_Mordred.csv", index_col=0)
        df = df.reindex(peptides)
        return df
    if os.path.exists("Tetrapeptide_Mordred.csv") and len(peptides[0]) == 4:
        peptides = [pu.translate1to3(x) for x in peptides]
        df = pandas.read_csv("Tetrapeptide_Mordred.csv", index_col=0)
        df = df.reindex(peptides)
        return df
    
    calc = Calculator(descriptors, ignore_3D=True)
    mols = []
    for pep in peptides:
        try:
            mol = makepeptide(pep)
            mol = Chem.AddHs(Chem.RemoveHs(mol))
            mols.append(mol)
        except AttributeError:
            print("FAILED To makepeptide of", pep)
    
    df = calc.pandas(mols, nproc = CPUS)
    df = df[keys]

    threeletterpeptides = []
    for pep in peptides:
        threeletterpeptides.append(pu.translate1to3(pep))
    df.index = threeletterpeptides
    return df

def calcLOGP(peptide):
    if "-" in peptide:
        peptide = pu.translate3to1(peptide)
    letters_1 = ["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y"]
    Gwif = [0.17, -0.24, 1.23, 2.02, -1.13, 0.01, 0.17, -0.31, 0.99, -0.56, -0.23, 0.42, 0.45, 0.58, 0.81, 0.13, 0.14, 0.07, -1.85, -0.94, ] #kcal / mol
    Gwoct = [0.5, -0.02, 3.64, 3.63, -1.71, 1.15, 0.11, -1.12, 2.8, -1.25, -0.67, 0.85, 0.14, 0.77, 1.81, 0.46, 0.25, -0.46, -2.09, -0.71, ] #kcal / mol
    LogP = 0
    for AA in list(peptide):
        index = letters_1.index(AA)
        LogP += Gwif[index] - Gwoct[index]
    return LogP

def restart(fname):
    content = readin(fname)
    restart_peps = []
    restart_APs = []
    for line in content.split("\n"):
        if len(line.split()) == 3:
            line = line.split()
            restart_peps.append(line[0])
            restart_APs.append(float(line[2]))
    return restart_peps, restart_APs

In [28]:
if __name__ == "__main__":
    Seed = 4364667
    random.seed(Seed)
    np.random.seed(Seed)

    argv = [1,3,4,100]
    #
    #for i in range(4):
    #    e = int(input())
    #    argv.append(e)
    
    if len(sys.argv) != 4:
        print("""Please pass argv's, ie.
              python Act*M*.py nCPUS nAmino_acids Max_logP Iterations {MODE}""")
    nJobs       = int(argv[0])
    AminoAcids  = int(argv[1])
    Max_logP    = int(argv[2])
    ITER    = int(argv[3])
    
    MODE = None

    
    APs = pandas.read_csv("tripeptide_AP.txt", sep=": ", index_col=0, header=None, engine = 'python')
    LetterLength = (4*AminoAcids)-1
    APs = APs.reindex([x for x in APs.index if len(x) == LetterLength])
    APs = APs.sort_values(1)
    N_top_peptides =  math.ceil((np.log(AminoAcids)**2) * 1000)
    TopPeptides = APs.nlargest(N_top_peptides, 1)
    GenStep = 10
    SVR_Features = np.load("SVR_Features_Selected1.npy")
    
    TopFolder = pu.Num2Word[AminoAcids].lower()+"peptides"
    InitialTrainingSet = ["".join(["ALA-" for x in range(AminoAcids)])[:-1]]
    outf = "New"+pu.Num2Word[AminoAcids]+"peptides_LOGP"+str(Max_logP)+"_N=1_SVRrbf.txt" 
    if MODE == "RANGE":
        outf = outf.replace(".txt", "_MODE=RANGE.txt")
        print("WARNING: MODE = RANGE")
        time.sleep(10)
    print(TopFolder, InitialTrainingSet, outf)

    




Please pass argv's, ie.
              python Act*M*.py nCPUS nAmino_acids Max_logP Iterations {MODE}
tripeptides ['ALA-ALA-ALA'] NewTripeptides_LOGP4_N=1_SVRrbf.txt


In [29]:
for pep in InitialTrainingSet:
        if pep not in APs.index:
            print("Peptide(s) missing from APs file", pep)
            sys.exit()
print("All peptides from sample found in APs file")

All peptides from sample found in APs file


In [30]:
if os.path.exists(outf): ### Restart
    print("Restarting from:", outf)
    restart_peps, restart_APs = restart(outf)
    print([pu.translate3to1(x) for x in restart_peps])
    Writeout = open(outf, 'a')
    Restarting = True
    TOTAL = len(restart_peps)
else:
    Restarting = False
    TOTAL = 0
if not os.path.exists(outf):
    restart_peps = []
    Writeout = open(outf, 'w')
    
new_AP_predictions = [-1] * len(InitialTrainingSet)
X_train = gen([pu.translate3to1(x) for x in InitialTrainingSet + restart_peps], SVR_Features, nJobs)
y_train = APs.reindex(X_train.index)
for i in range(len(restart_peps)):
    y_train.loc[restart_peps[i]] = restart_APs[i]


Restarting from: NewTripeptides_LOGP4_N=1_SVRrbf.txt
[]


KeyError: "['SPRatio', 'NH2', 'S', 'LogP WW', 'Z', 'MaxASA', 'Bulkiness', 'OH'] not in index"

In [31]:
pre = prescreen.prescreen(ap_data=None, load_data = False, model_type="rbf")
    
pre.nJobs = nJobs
pre.load_unknown(pu.Num2Word[AminoAcids].lower()+"peptides.hdf5")
pre.move_data(list(X_train.index))
pre.y_train = y_train
pre.removeLogP(LogP = Max_logP)
pre.X_test = normalize_pandas_data(pre.X_test) 
if Restarting:
    print("Removing peptides from test step that were found in previous runs")
    pre.remove_pep(restart_peps)

print("Prescreening...")
N_top_peptides =  math.ceil((np.log(AminoAcids)**2) * 1000) #np.log is ln(), np.log10 is log10()
screened = pre.get_top_sample(N_top_peptides)
for i in range(len(screened)):
    screened[i] = pu.translate3to1(screened[i])
X_test = gen(screened, SVR_Features, nJobs)
drop = [p for p in X_train.index if p in X_test.index]
X_test = X_test.drop(drop)

KeyError: 'LogP WW'

In [32]:
import h5py

h5_file = h5py.File('tripeptides.hdf5','r')
names = h5_file["peptides"][()]
data = h5_file["data"][()]
Features = h5_file["features"][()]
h5_file.close()


x = pandas.DataFrame(data, index=names, columns=Features)
x


,b'SP2',b'NH2',b'MW',b'S',b'LogP WW',b'Z',b'MaxASA',b'RotRatio',b'Bulkiness',b'OH'
b'ALA-ALA-CYS',0.0,0.0,263.329437,1.0,-0.88,0.0,425.0,0.000000,36.459999,0.0
b'ALA-ALA-ASP',1.0,0.0,275.279449,0.0,-3.07,-1.0,451.0,0.333333,34.680000,0.0
b'ALA-ALA-GLU',1.0,0.0,289.299438,0.0,-2.27,-1.0,481.0,0.250000,36.570000,0.0
b'ALA-ALA-PHE',6.0,0.0,307.359436,0.0,-0.08,0.0,498.0,2.000000,42.799999,0.0
b'ALA-ALA-GLY',0.0,0.0,217.239441,0.0,-1.80,0.0,362.0,0.000000,26.400000,0.0
...,...,...,...,...,...,...,...,...,...,...
b'TYR-TYR-THR',12.0,0.0,445.469452,0.0,-0.57,0.0,698.0,3.000000,51.830002,3.0
b'TYR-TYR-VAL',12.0,0.0,443.499451,0.0,0.07,0.0,700.0,2.400000,57.630001,2.0
b'TYR-TYR-TRP',20.0,0.0,530.579407,0.0,-0.22,0.0,811.0,6.666667,57.730000,2.0
b'TYR-TYR-TYR',18.0,0.0,507.539459,0.0,-0.69,0.0,789.0,6.000000,54.090000,3.0


In [33]:
print(X_train)
print('*')
print(list(X_train))
print('*')
print(X_train.index)
print('*')
print(list(X_train.index))

                b'SP2'  b'NH2'       b'MW'  b'S'  b'LogP WW'  b'Z'  b'MaxASA'  \
b'ALA-ALA-CYS'     0.0     0.0  263.329437   1.0       -0.88   0.0      425.0   
b'ALA-ALA-ASP'     1.0     0.0  275.279449   0.0       -3.07  -1.0      451.0   
b'ALA-ALA-GLU'     1.0     0.0  289.299438   0.0       -2.27  -1.0      481.0   
b'ALA-ALA-PHE'     6.0     0.0  307.359436   0.0       -0.08   0.0      498.0   
b'ALA-ALA-GLY'     0.0     0.0  217.239441   0.0       -1.80   0.0      362.0   
...                ...     ...         ...   ...         ...   ...        ...   
b'TYR-TYR-THR'    12.0     0.0  445.469452   0.0       -0.57   0.0      698.0   
b'TYR-TYR-VAL'    12.0     0.0  443.499451   0.0        0.07   0.0      700.0   
b'TYR-TYR-TRP'    20.0     0.0  530.579407   0.0       -0.22   0.0      811.0   
b'TYR-TYR-TYR'    18.0     0.0  507.539459   0.0       -0.69   0.0      789.0   
b'ALA-ALA-ALA'     0.0     0.0  231.269424   0.0       -0.99   0.0      387.0   

                b'RotRatio'

In [34]:
X_train = x.reindex(list(x.index))
x = x.drop(list(X_train.index))